#### Lecture de la bd

#### Lecture de la bd

lien vers le jeu de données initial https://www.kaggle.com/datasets/Microsoft/microsoft-security-incident-prediction?select=GUIDE_Train.csv

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


puisque le fichier initial est volumineux, nous avons décider d'en, extraire un echantillon alléatoire
composé de 1% de la donnée initiale, qui sera notre nouvelle source des données.

La lecture se fera par blocks pour éviiter de charger le fichier en entier

In [ ]:
def echantillonner_stratifie(chemin, frac=0.01, cible='y'):
    sample_list = []

    for chunk in pd.read_csv(chemin, chunksize=10000):
        chunk = chunk.dropna(subset=[cible])

        # Échantillonnage stratifié
        sample = (
            chunk
            .groupby(cible, group_keys=False)
            .apply(lambda x: x.sample(frac=frac, random_state=2026))
        )

        sample_list.append(sample)

    df_sample = pd.concat(sample_list)

    df_sample.to_csv("../echantillon.csv", index=False)

#echantillonner_stratifie("../GUIDE.csv", frac=0.01, cible='IncidentGrade')

In [ ]:
df = pd.read_csv("../echantillon.csv", low_memory= False)

print(f'La base des données a {df.shape[0]} enrégistrements et {df.shape[1]} colonnes. Les colonnes sont : ')
df.columns

Parmis ces colonnes, beaucoup sont des id, on va les supprimer avant ttout traitement

In [ ]:
colones_exploitables = ['Timestamp','Category', 'MitreTechniques','IncidentGrade','ActionGrouped', 'ActionGranular', 'EntityType', 'EvidenceRole',
                  'RegistryValueData','ThreatFamily','ResourceType','Roles','OSFamily','OSVersion','AntispamDirection','SuspicionLevel',
                  'LastVerdict', 'CountryCode','State', 'City']

df_= df[colones_exploitables]
df_.head()

In [ ]:
valeurs_manquantes = df_.isna().sum()

# Pourcentage de valeurs manquantes
pourcentage_nan = (valeurs_manquantes / len(df_)) * 100

# Création du DataFrame final
valeurs_manquantes = pd.DataFrame({
    'Column': valeurs_manquantes.index,
    'Missing_Values': valeurs_manquantes.values,
    'Pourcentage_NaN': pourcentage_nan.values
})

valeurs_manquantes = (
    valeurs_manquantes.sort_values(by='Pourcentage_NaN', ascending=False)
)

print(valeurs_manquantes)

* On va supprimer les colones 'ResourceType','ActionGrouped', 'ActionGranular ', ' ThreatFamily ', 'AntispamDirection ' et ' Roles'

* Dans la colonne ‘SuspicionLevel’ , la valeur manquante signifie que la transaction est non suspecte ou non évaluée. 

* La valeur manquante dans la colone ‘LastVerdict’ signifie qu’il n’y a pas encore de verdict, ou que le verdict est inconnu

* Dans la colonne ‘MitreTechniques’ On va remplacer les valeurs manquantes par “Unknown”

In [ ]:
colones_utiles =  ['Timestamp','Category', 'MitreTechniques','IncidentGrade',  'EntityType', 'EvidenceRole',
                  'RegistryValueData','OSFamily','OSVersion','SuspicionLevel',
                  'LastVerdict', 'CountryCode','State', 'City']

In [ ]:
df_ = df[colones_utiles]
df_.info()

In [ ]:
df_ = df_.copy()

colonnes_obj = [
    'RegistryValueData', 'OSFamily', 'OSVersion', 'SuspicionLevel',
    'LastVerdict', 'CountryCode', 'State', 'City'
]

for col in colonnes_obj:
    df_[col] = df_[col].astype('object')

print(df_.dtypes)

In [ ]:
#On visualise la variable cible
technique_counts = df_.groupby('IncidentGrade').size().reset_index(name='Count')

# Trier décroissant
Incident_grade  = technique_counts.sort_values(by='Count', ascending=False)


plt.figure()
sns.barplot(data=Incident_grade, x='IncidentGrade', y='Count')

plt.title("Incident_grade")
plt.xlabel("")
plt.ylabel("")

plt.show()